# Phase 3 — Rule-based arm

Answers the research question (`paper/PLAN.md`) with an **explicit,
hand-written** operation sequence, calling `smart_spatial_system` 0.3.0's
real plugin functions directly — no LLM, no planner. Arm 2
(`03_llm_arm.ipynb`) answers the same question by handing a natural-language
version of it to `s3geo.query()` and letting the planner choose. Neither
arm is allowed to see the other's plan (`CLAUDE.md`).

**⚠️ Not yet executed.** Written and reviewed without running it, for the
same reason as `01_data_and_problem.ipynb`: this session has no local
install of the pinned package or its geospatial dependencies. Every
function signature, default, and behaviour used below was read directly
from the real 0.3.0 source (`orchestrator/planning/op_catalog.py` and the
individual `plugins/*.py` files) — see `paper/PLAN.md` "Arm 1" and
`CLAUDE.md` "Verified against 0.3.0 source" for the citations. **Run every
cell top to bottom** and read the printed output before trusting anything
in `results/`. If a check fails or a call behaves differently than
documented below, stop and write it up in `bugs/` per `CLAUDE.md` — do not
silently work around it.

This notebook also resolves two items `paper/PLAN.md` "Open decisions"
left open, empirically, in place, rather than by argument:

1. **Which counting operation** (`filter_points_in_polygon` vs.
   `spatial_join`) for facilities-per-mahalle — §5 below.
2. **Whether the CRS-mismatch safety net on `find_nearest_neighbors`
   still exists at 0.3.0** (Vienna's candidate Bug 5) — §3 below.


In [ ]:
import json
from pathlib import Path

import geopandas as gpd
import pandas as pd

from geochat_sdk.types.vector import VectorOut
from plugins.crs_transformer import transform_vector_crs
from plugins.nearest_neighbor import find_nearest_neighbors
from plugins.spatial_join import spatial_join_features
from plugins.dissolve_aggregator import dissolve_features
from plugins.feature_scoring import rank_features
from plugins.report_builder import build_report

RAW = Path("../data/raw")
RESULTS = Path("../results")
RESULTS.mkdir(parents=True, exist_ok=True)

SOURCE_CRS = "EPSG:4326"
ANALYSIS_CRS = "EPSG:32635"  # WGS 84 / UTM zone 35N — see data/README.md
THRESHOLDS_M = [1000, 1500, 2000]
HEADLINE_THRESHOLD_M = 2000


def gdf_to_vectorout(gdf: gpd.GeoDataFrame) -> VectorOut:
    """GeoDataFrame -> VectorOut, the input shape every plugin below expects.
    Plugins take VectorOut / FeatureCollection dict / Feature dict / list —
    never a GeoDataFrame directly (confirmed reading plugins/nearest_neighbor.py).
    """
    return VectorOut.from_geopandas(gdf)


def features_to_gdf(features: list[dict], crs: str) -> gpd.GeoDataFrame:
    """VectorOut.features (plain GeoJSON Feature dicts) -> GeoDataFrame.
    gpd.GeoDataFrame.from_features() does not carry a CRS — every plugin
    output is CRS-naive GeoJSON, so the caller (us) must set it explicitly
    based on what we know the data actually is, not what a plugin claims.
    """
    if not features:
        return gpd.GeoDataFrame(columns=["geometry"], geometry="geometry", crs=crs)
    gdf = gpd.GeoDataFrame.from_features(features)
    return gdf.set_crs(crs, allow_override=True)


## 1. Load raw layers and defensively re-check cleanliness

This arm loads `data/raw/` directly (EPSG:4326), **not**
`data/processed/`, so that §2 genuinely exercises the framework's own
`crs_transform` op end to end — reusing `data/processed/` (already
reprojected by `01_data_and_problem.ipynb`'s plain `geopandas.to_crs()`)
would make step 1 of `paper/PLAN.md`'s Arm 1 methodology a no-op.

`01_data_and_problem.ipynb`'s real run already found zero null geometries,
zero invalid mahalle rings, and zero unnamed mahalle in this exact data
(`paper/PLAN.md` "Findings"). We re-assert that here rather than trust it
blindly — a failure would mean something about the data changed since that
run, which should stop this notebook, not be silently patched over.


In [ ]:
hospitals_raw = gpd.read_file(RAW / "hospitals.geojson")
mahalle_raw = gpd.read_file(RAW / "mahalle_boundaries.geojson")

assert hospitals_raw.crs is not None and hospitals_raw.crs.to_string().upper() == "EPSG:4326", hospitals_raw.crs
assert mahalle_raw.crs is not None and mahalle_raw.crs.to_string().upper() == "EPSG:4326", mahalle_raw.crs

null_hosp = hospitals_raw.geometry.isna().sum()
null_mah = mahalle_raw.geometry.isna().sum()
invalid_mah = (~mahalle_raw.geometry.is_valid).sum()
unnamed_mah = mahalle_raw["name"].isna().sum() if "name" in mahalle_raw.columns else len(mahalle_raw)

print(f"hospitals_raw: {len(hospitals_raw)} features, {null_hosp} null geometry")
print(f"mahalle_raw:   {len(mahalle_raw)} features, {null_mah} null geometry, "
      f"{invalid_mah} invalid, {unnamed_mah} unnamed")

if null_hosp or null_mah or invalid_mah or unnamed_mah:
    raise ValueError(
        "data/raw/ is not as clean as 01_data_and_problem.ipynb found it to be — "
        "stop and re-run/inspect phase 2 before trusting this notebook's output."
    )

# No 'ilçe' (district) column: the Overpass admin_level=8 relations fetched
# here carry only osm_id/osm_type/admin_level/name(+incidental tags like
# postal_code, wikidata) — no parent-district tag. Verified directly against
# data/raw/mahalle_boundaries.geojson's actual properties. paper/PLAN.md's
# "Expected outputs" table lists an 'ilçe' column; that column is dropped
# from results/rule_based_underserved.csv below and PLAN.md should be
# updated to match rather than the notebook inventing a district it doesn't
# have.
print("mahalle_raw columns:", sorted(mahalle_raw.columns))


## 2. Step 1 — `crs_transform` each layer individually (4326 → 32635)

Op-catalog `crs_transform` → `transform_vector_crs`
(`plugins/crs_transformer.py`). **Its default `target_crs` is
`EPSG:3857`**, not our analysis CRS — `target_crs` must be passed
explicitly on every call, never relied on as a default. `allowed_crs` in
`config/plugins/crs_transformer.yaml` is `[]` (unrestricted) by default, so
EPSG:32635 needs no extra allow-listing.

Cross-checked below against `01_data_and_problem.ipynb`'s real, independent
reprojection (`geopandas.to_crs()`) of the exact same source data — two
different reprojection code paths over the same 964 mahalle should land on
the same bounds. If they don't, that's worth its own `bugs/` report.


In [ ]:
hospitals_t = transform_vector_crs(
    gdf_to_vectorout(hospitals_raw),
    source_crs=SOURCE_CRS,
    target_crs=ANALYSIS_CRS,
    engine="auto",
)
mahalle_t = transform_vector_crs(
    gdf_to_vectorout(mahalle_raw),
    source_crs=SOURCE_CRS,
    target_crs=ANALYSIS_CRS,
    engine="auto",
)

hospitals = features_to_gdf(hospitals_t.features, ANALYSIS_CRS)
mahalle = features_to_gdf(mahalle_t.features, ANALYSIS_CRS)

assert hospitals.crs.to_string().upper() == "EPSG:32635"
assert mahalle.crs.to_string().upper() == "EPSG:32635"
assert len(hospitals) == len(hospitals_raw)
assert len(mahalle) == len(mahalle_raw)

bounds = mahalle.total_bounds  # minx, miny, maxx, maxy
print(f"mahalle bounds after transform_vector_crs: {bounds}")

# 01_data_and_problem.ipynb's real run reported x: 581527-748500, y: 4519307-4604146
# (paper/PLAN.md 'Findings'). transform_vector_crs's own coordinate_precision
# rounding (config/plugins/crs_transformer.yaml) can differ slightly from
# geopandas' pyproj call, so this is a sanity band, not an exact-equality check.
expected = (581527, 4519307, 748500, 4604146)
tolerance_m = 1000
for got, exp, label in zip(bounds, expected, ["minx", "miny", "maxx", "maxy"]):
    diff = abs(got - exp)
    status = "OK" if diff <= tolerance_m else "MISMATCH"
    print(f"  {label}: transform_vector_crs={got:.1f}  geopandas.to_crs={exp}  diff={diff:.1f}m  [{status}]")


## 3. Smoke test — CRS-mismatch safety net on `find_nearest_neighbors`

`plugins/distance_calculator.py`'s `_raise_if_crs_mismatch()` (imported by
`nearest_neighbor.py`) is **opt-in**: it only fires when *both*
`source_crs` and `target_crs` are supplied, and it is a **string comparison
of the hints you pass**, not an inspection of the actual coordinates — it
cannot catch mislabelled-but-consistent data, only inconsistent labels.
Confirming this behaves as documented resolves `paper/PLAN.md`'s open
question about whether Vienna's candidate CRS-mismatch bug is still present
at 0.3.0.


In [ ]:
# (a) Deliberately mismatched hints on data that is actually consistent
# (both layers really are EPSG:32635 at this point) — should raise on the
# label mismatch alone, per the docstring in plugins/distance_calculator.py.
crs_mismatch_raised = False
try:
    find_nearest_neighbors(
        source_features=gdf_to_vectorout(mahalle.head(5)),
        target_features=gdf_to_vectorout(hospitals.head(5)),
        k=1,
        source_crs="EPSG:4326",   # wrong on purpose
        target_crs=ANALYSIS_CRS,  # correct
    )
except ValueError as exc:
    crs_mismatch_raised = True
    print("Raised as expected:", exc)

if not crs_mismatch_raised:
    print("⚠️ Did NOT raise on mismatched CRS hints — this contradicts "
          "plugins/distance_calculator.py's own docstring. File a bugs/ report.")

# (b) Matching hints on the same small sample — should succeed normally.
_ = find_nearest_neighbors(
    source_features=gdf_to_vectorout(mahalle.head(5)),
    target_features=gdf_to_vectorout(hospitals.head(5)),
    k=1,
    source_crs=ANALYSIS_CRS,
    target_crs=ANALYSIS_CRS,
)
print("Matching-hint call succeeded with no exception raised, as expected.")


## 4. Step 2 — nearest-neighbor distance: boundary and centroid

Op-catalog `spatial_nearest` → `find_nearest_neighbors`
(`plugins/nearest_neighbor.py`), `k=1`. Two passes, both reported per
`CLAUDE.md` "Distance semantics":

- **boundary**: mahalle polygon → nearest hospital. A mahalle containing a
  hospital gets `0.0` — correct GIS behaviour, but it means dense mahalle
  tie at the top.
- **centroid**: mahalle centroid point → nearest hospital. The more honest
  figure for a health-access question, used for thresholding below.

9 mahalle have a centroid falling outside their own polygon (island /
irregular coastline shapes — Kınalıada, etc.; `paper/PLAN.md` "Findings").
Their centroid distance is flagged, not silently used as if it meant the
same thing it does for a normal mahalle.


In [ ]:
# --- boundary distance ---
result_boundary = find_nearest_neighbors(
    source_features=gdf_to_vectorout(mahalle),
    target_features=gdf_to_vectorout(hospitals),
    k=1,
    source_crs=ANALYSIS_CRS,
    target_crs=ANALYSIS_CRS,
    distance_field="dist_boundary_m",
)
boundary_gdf = features_to_gdf(result_boundary.features, ANALYSIS_CRS)
assert len(boundary_gdf) == len(mahalle), "expected one nearest-neighbor row per mahalle (k=1)"

# --- centroid distance ---
CENTROID_OUTSIDE_POLYGON_NAMES = {
    "Mimar Kemalettin", "Fatih", "Orhanlı", "Malkoçoğlu", "Şamlar",
    "Kınalıada", "Maden", "Esenkent", "Karaburun",
}  # see paper/PLAN.md 'Findings' (01_data_and_problem.ipynb's real run)

mahalle_centroids = mahalle.copy()
mahalle_centroids["geometry"] = mahalle.geometry.centroid

result_centroid = find_nearest_neighbors(
    source_features=gdf_to_vectorout(mahalle_centroids),
    target_features=gdf_to_vectorout(hospitals),
    k=1,
    source_crs=ANALYSIS_CRS,
    target_crs=ANALYSIS_CRS,
    distance_field="dist_centroid_m",
)
centroid_gdf = features_to_gdf(result_centroid.features, ANALYSIS_CRS)
assert len(centroid_gdf) == len(mahalle), "expected one nearest-neighbor row per mahalle (k=1)"

dist_df = pd.DataFrame({
    "osm_id": boundary_gdf["osm_id"].values,
    "name": boundary_gdf["name"].values,
    "dist_boundary_m": boundary_gdf["dist_boundary_m"].values,
    "dist_centroid_m": centroid_gdf["dist_centroid_m"].values,
})
dist_df["centroid_outside_polygon"] = dist_df["name"].isin(CENTROID_OUTSIDE_POLYGON_NAMES)

print(f"{len(dist_df)} mahalle scored")
print(f"mahalle with dist_boundary_m == 0.0 (contain a facility): {(dist_df['dist_boundary_m'] == 0.0).sum()}")
print(dist_df.describe(include='all'))


## 5. Step 3 — facility count per mahalle: `spatial_join`, not `filter_points_in_polygon`

**Resolves `paper/PLAN.md`'s open decision.** Both ops are real and
catalog-registered, but only one can actually produce a per-mahalle count:

- `filter_points_in_polygon` (`plugins/spatial_predicate.py`) takes a
  *set* of polygons as one undifferentiated containment mask and returns
  which points fall inside *any* of them — it does not report **which**
  polygon matched. Read from source: no polygon identity survives into
  the output feature. Getting a per-mahalle breakdown from it would mean
  calling it once per mahalle (964 calls) — not how the op is meant to be
  used, and not a real op-catalog usage pattern.
- `spatial_join_features` (`plugins/spatial_join.py`) attaches the
  matched target (mahalle) feature to each source (hospital) feature —
  `_target_index` (position in the mahalle list passed in) and, with
  `include_target_properties=True`, the mahalle's own properties nested
  under `_joined_target_properties`. One call gives every hospital its
  containing mahalle; a `groupby` in this notebook turns that into counts.

`join_type="left"` (not the default `"inner"`) so a hospital matching no
mahalle (e.g. a coastline point just outside the AOI's administrative
polygon) is kept and counted as unmatched, not silently dropped —
`CLAUDE.md`'s rule against quietly losing data. `cardinality="first"`
(the default) is correct here: administrative mahalle polygons don't
overlap, so a point can only be `within` one.


In [ ]:
mahalle_reset = mahalle.reset_index(drop=True)  # _target_index below indexes into this exact order

join_result = spatial_join_features(
    source_features=gdf_to_vectorout(hospitals),
    target_features=gdf_to_vectorout(mahalle_reset),
    predicate="within",
    join_type="left",
    cardinality="first",
    include_target_properties=True,
    source_crs=ANALYSIS_CRS,
)

join_props = [f["properties"] for f in join_result.features]
join_df = pd.DataFrame(join_props)

n_matched = (join_df["_join_status"] == "matched").sum()
n_unmatched = (join_df["_join_status"] == "unmatched").sum()
print(f"hospitals matched to a mahalle: {n_matched}")
print(f"hospitals matched to NO mahalle: {n_unmatched}")
if n_unmatched:
    print(join_df.loc[join_df["_join_status"] == "unmatched", ["osm_id", "name"]].to_string(index=False))

matched = join_df[join_df["_join_status"] == "matched"].copy()
matched["mahalle_osm_id"] = matched["_target_index"].apply(lambda i: mahalle_reset.iloc[int(i)]["osm_id"])

facility_counts = (
    matched.groupby("mahalle_osm_id").size().rename("facility_count").reset_index()
)

print(f"mahalle with >=1 facility inside them: {len(facility_counts)} / {len(mahalle)}")
print(facility_counts["facility_count"].describe())


## 6. Step 4 — threshold at 1000 / 1500 / 2000 m (centroid distance)

Per `CLAUDE.md` "Threshold": report the underserved set at all three
distances so the headline number is visibly a function of the threshold,
not a fixed fact.

No `ilçe` column — see §1: the fetched data does not carry a
parent-district tag (verified against `data/raw/mahalle_boundaries.geojson`'s
real properties), so it is dropped here rather than invented. If a district
breakdown turns out to matter for the paper, it needs its own admin_level=6
fetch and a spatial join against it — out of scope for this notebook.


In [ ]:
results = dist_df.merge(
    facility_counts.rename(columns={"mahalle_osm_id": "osm_id"}),
    on="osm_id", how="left",
)
results["facility_count"] = results["facility_count"].fillna(0).astype(int)

for t in THRESHOLDS_M:
    results[f"underserved_{t}"] = results["dist_centroid_m"] > t

print(results[["name", "dist_boundary_m", "dist_centroid_m", "facility_count"] +
              [f"underserved_{t}" for t in THRESHOLDS_M]].head())
print()
for t in THRESHOLDS_M:
    n = int(results[f"underserved_{t}"].sum())
    print(f"underserved at {t} m: {n} / {len(results)} mahalle ({100 * n / len(results):.1f}%)")


## 7. Step 5 — dissolve underserved mahalle into contiguous regions

**Structural asymmetry, not a bug** (`paper/PLAN.md` Arm 1 step 5,
`CLAUDE.md`): `dissolve_features` (`plugins/dissolve_aggregator.py`) is a
real, registered capability but is confirmed absent from
`orchestrator/planning/op_catalog.py`'s `OP_CATALOG`, so it is not a
planner-reachable `op_name` — `s3geo.query()` can never produce this step,
however good its prompt. Arm 1 is allowed to call the plugin function
directly, bypassing the planner; Arm 2 structurally cannot ask for it.
Name this explicitly in the paper — it is a capability gap between the two
arms, not a fair point to score the LLM arm down on.

`group_by=None` dissolves **all** input features into one output feature
via `shapely.ops.unary_union` — for a set of polygons that includes both
touching and non-touching ones, `unary_union` merges the touching ones and
keeps the rest as separate parts of one `MultiPolygon`, which is exactly
"contiguous underserved regions" as one dissolve call. `.explode()`
afterward turns that single multi-part feature into one row per contiguous
region, so we can report how many distinct clusters there are.


In [ ]:
underserved_2000 = mahalle_reset.merge(
    results[["osm_id", f"underserved_{HEADLINE_THRESHOLD_M}"]],
    on="osm_id", how="inner",
)
underserved_2000 = underserved_2000[underserved_2000[f"underserved_{HEADLINE_THRESHOLD_M}"]].copy()
print(f"{len(underserved_2000)} mahalle underserved at {HEADLINE_THRESHOLD_M} m, going into dissolve")

if len(underserved_2000) == 0:
    print("No underserved mahalle at the headline threshold — nothing to dissolve. "
          "underserved_regions ends up empty; that is a real result, not an error.")
    underserved_regions = gpd.GeoDataFrame(columns=["geometry"], geometry="geometry", crs=ANALYSIS_CRS)
else:
    dissolve_result = dissolve_features(
        gdf_to_vectorout(underserved_2000),
        group_by=None,
        engine="auto",
        source_crs=ANALYSIS_CRS,
    )
    dissolved_gdf = features_to_gdf(dissolve_result.features, ANALYSIS_CRS)
    underserved_regions = dissolved_gdf.explode(index_parts=False).reset_index(drop=True)

print(f"{len(underserved_regions)} contiguous underserved region(s) at {HEADLINE_THRESHOLD_M} m")


## 8. Step 6 — `build_report`

Op-catalog `build_report` → `build_report` (`plugins/report_builder.py`).
It expects a **ranked/scored** feature set (`score_field`/`rank_field`), so
`rank_features` (`plugins/feature_scoring.py`) runs first, ranking
underserved mahalle by centroid distance descending — farthest-from-care
first. This is a distance ordering, not a value judgement; said explicitly
here because `build_report`'s default `report_spec` is written for a
real-estate "investment score" domain (`score_field` even defaults to
`"investment_score"`) — worth naming in the paper as a rough edge for
reuse outside that domain, not a bug, since `report_spec` is fully
overridable.


In [ ]:
underserved_for_report = underserved_2000.merge(
    results[["osm_id", "dist_centroid_m", "dist_boundary_m", "facility_count"]],
    on="osm_id", how="left", suffixes=("", "_r"),
)

if len(underserved_for_report) == 0:
    report_result = None
    print("No underserved mahalle to rank/report at the headline threshold.")
else:
    ranked_result = rank_features(
        gdf_to_vectorout(underserved_for_report),
        score_field="dist_centroid_m",
        rank_field="rank",
        descending=True,
    )
    report_result = build_report(
        ranked_result,
        report_spec=None,  # default spec — see note above on its real-estate framing
        score_field="dist_centroid_m",
        rank_field="rank",
        name_field="name",
        metadata={"study": "istanbul-health-access", "threshold_m": HEADLINE_THRESHOLD_M},
    )
    print("build_report success:", report_result.success)
    if not report_result.success:
        print("errors:", report_result.errors)
    print("table title:", report_result.table.get("title"))
    print("table rows:", report_result.table.get("total_rows"))
    print(json.dumps(report_result.summary, indent=2, ensure_ascii=False)[:2000])


## 9. Write outputs

Per `paper/PLAN.md` "Expected outputs" (minus the `ilçe` column — §6).


In [ ]:
results.to_csv(RESULTS / "rule_based_underserved.csv", index=False)
print(f"wrote {RESULTS / 'rule_based_underserved.csv'} ({len(results)} rows)")

if len(underserved_regions):
    underserved_regions.to_file(RESULTS / "underserved_regions.geojson", driver="GeoJSON")
    print(f"wrote {RESULTS / 'underserved_regions.geojson'} ({len(underserved_regions)} regions)")
else:
    print("no underserved regions to write")

if report_result is not None:
    report_path = RESULTS / "rule_based_report.json"
    report_path.write_text(
        json.dumps(
            {
                "meta": report_result.meta,
                "summary": report_result.summary,
                "table": report_result.table,
                "success": report_result.success,
                "errors": report_result.errors,
            },
            indent=2, ensure_ascii=False,
        ),
        encoding="utf-8",
    )
    print(f"wrote {report_path}")


## 10. Summary — copy this cell's real output into `paper/PLAN.md`


In [ ]:
summary = {
    "mahalle_scored": len(results),
    "underserved_by_threshold": {
        str(t): int(results[f"underserved_{t}"].sum()) for t in THRESHOLDS_M
    },
    "contiguous_underserved_regions_at_headline": len(underserved_regions),
    "hospitals_unmatched_in_spatial_join": int(n_unmatched),
    "mahalle_with_zero_facilities": int((results["facility_count"] == 0).sum()),
    "crs_mismatch_safety_net_raised_as_expected": bool(crs_mismatch_raised),
    "crs_transform_vs_geopandas_bounds_cross_check": "see §2 output above",
    "analysis_crs": ANALYSIS_CRS,
    "headline_threshold_m": HEADLINE_THRESHOLD_M,
}
print(json.dumps(summary, indent=2, ensure_ascii=False))
